# 9주차 실습 — 의료데이터 SQL 종합실습

- **데이터**: `aact_mini` — 임상시험 37,509건(2015~2024년 시작). 7 · 8주차와 같은 데이터입니다.
- **오늘 배우는 것**
  1. 분석 질문을 다섯 단계로 나누어 SQL로 바꿀 수 있다
  2. 서브쿼리로 대상을 먼저 뽑고 그 안에서 셀 수 있다
  3. 조건을 하나씩 더하며 결과를 확인할 수 있다
  4. 검산 다섯 가지로 결과를 스스로 점검할 수 있다
  5. 과제 1을 제출할 수 있다 (5절 안내)
- **사용법**: 위에서부터 차례로 실행합니다. 예제 셀은 실행만 하면 됩니다 (2절 1~4단계는 9주차 슬라이드의 SQL 그대로입니다). '직접 해보기'에서 `___` 를 채운 뒤 `check(...)` 로 확인합니다.
- **점수는 없습니다.** 과제 1은 따로 제출합니다.

## 0. 시작하기 전에

- **Colab 에서 열었다면 먼저 메뉴의 `파일 → Drive에 사본 저장`** 을 누르세요. 그래야 작성한 내용이 남습니다.
- 셀 실행: 셀 왼쪽의 ▶ 단추 또는 `Shift + Enter`. **위에서부터 차례로** 실행합니다.
- `%%sql` 로 시작하는 셀에는 SQL만 적습니다. `%%sql` 바로 아래의 `-- Q1` 같은 줄은 지우지 마세요 (문항 이름표입니다).
- 오류가 나면 붉은 글씨 중 **`Error:` 가 들어간 줄**만 읽으면 됩니다. 연결이 이상하면 `런타임 → 런타임 다시 시작` 후 준비 셀부터 다시 실행하세요.

In [ ]:
#@title ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 학번 셀과 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "바로 위 SQL 셀에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 SQL을 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, f"결과가 표가 아닙니다. %%sql 셀을 실행한 뒤, 바로 다음 셀에서 check('{name}', _) 를 실행하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 check 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

In [ ]:
#@title 자동 확인 도우미 — 정답 SQL 은 들어 있지 않습니다 (수정하지 마세요) { display-mode: "form" }
ANSWERS = _load_answers("eNqVlFFP2lAUx7/KSZ80MYYCBbpkD0v2aEx82NOyBwplMzGygMsejAloXQiwBCeVsrVEJ1MxmnUIExOe9lF87Dn9Dru3hVkIbuOpvefcc9v/7/7P2RbWROEJbAv59dfsKUQz8YgqhWQlLKeEJRCyj8Rz2fd5FhfZayq7wV9fClQx3YZBJUN4xcJv1je3eKXT01xjAKSXqFEjzYT7goHVAu0VwS8A7Jb4Drzs4Il1X2iSpQGeV/Gw7dwcLwHadefum1+M5To4t+O9o0MBD72EfxyrBFaHZ/tYKWGlvezJyKXVnJpm/5NJbuRVHnm7tZ7dTG6MQzssthYOosikY+GEkklFo5lIEEUkJcYiaigeS2akAIrEBIq9ortn8uxsKHhQA/fA9H7+8oKvqFXyNDR6C9Swna4N7m57EYtDPLsCx24yJUwV0HGRyv2RMsAbje6uIQ+/bkEBPDGBjnpY9s+2DH4aWQM6LzCIlttqzk8kMkEkIidTYkSWklI6SGQ6PtMcTQ1Pqz4NvocpdG6GTpeHJv3iJ+wC+EmsmWOjPAU/OcqQNSSrB3inMdUcIBdKepmswqiCWiMTwEKcTodkX8Dqi5UVcJs648KgdLCrL87DIxrkISYSipoSJTkamnDIdHwWD1c3nf412QNqXE1aY9dAawhMGZ5pXAp+tNnC1WzuFnb/bqsKZBQ9zd4m3hGso9j1LzEKB6wG6zYZPWaCcYdQvTgGQV/3Xb3DTijRsQlRrFw4zEXNAjcZ/5xjf5qLiBQkklbEkBpWpLAsSkEi0/H/Hx+ktzwnn9jOjx6/+1G/4HmLdI30Ibi6gd+5eni2+txrAsYWqK3xOcIIcVD4s/nHC8vgdydX+hhfjvNqQK3aA9sHhvPQiQXpyHE5HVdDibiiiEE60/GZdPomfajybv5S8+ZK3+RT1F8Hga2JwIE9jN0FMhmnOvxl6C56TeXfP+726HPHYxecNf/UvJV7p+7s/AbDUOQp")

## 1. 질문을 SQL로 바꾸기 — 다섯 단계 절차

| 단계 | 할 일 |
|---|---|
| ① 질문을 한 문장으로 | “무엇을, 어떤 대상에서, 몇 개” 를 분명히 적는다 |
| ② 세는 단위를 정한다 | 시험 건수? 환자 수? 기관 수? — 결과의 한 행이 무엇인가 |
| ③ 필요한 표와 연결 키 | 어느 표에 있는 값인가, 무엇으로 연결하는가 |
| ④ 조건을 하나씩 더한다 | 한 번에 다 쓰지 않고 건수 변화를 보며 추가 |
| ⑤ 검산한다 | 건수 · 분모 · 중복을 확인한 뒤 결과를 쓴다 |

SQL을 먼저 쓰지 말고, ①②③을 **말이나 글로 먼저** 적는 것이 가장 빠릅니다.

### 같은 질문, 다른 답 — 세는 단위를 정하지 않으면 생기는 일

“당뇨 임상시험이 몇 건인가요?” — 이 질문만으로는 SQL을 쓸 수 없습니다.

| | 이렇게 물으면 — 모호함 | 이렇게 물으면 — 분명함 |
|---|---|---|
| 질문 | 당뇨 시험이 몇 건인가요? | 2015~2024년에 시작한, MeSH 용어가 'Diabetes Mellitus, Type 2' 인, 중재 연구 중 완료된 것이 몇 **건**인가요? |
| 결과 | 제2형만? 제1형도? 진행 중인 것만? 완료된 것도? 어느 나라? 어느 기간? → 사람마다 다른 숫자가 나옴 | 누가 해도 같은 숫자 |

연구자 · 부서에서 분석을 요청받으면 먼저 이 네 가지를 되묻습니다: **대상 · 기간 · 조건 · 세는 단위**. 11주차 요청서에서 다시 봅니다.

### 서브쿼리 — 대상을 먼저 뽑고, 그 안에서 세기

괄호 안이 먼저 계산됩니다. 괄호 안에서 ‘대상자 목록’을 먼저 만들고, 바깥에서 그 목록에 드는 것만 셉니다.

먼저 괄호 안만 실행해 봅니다. 시험 번호(`nct_id`) 목록이 나옵니다.

In [ ]:
%%sql
-- 괄호 안만 실행 → 제2형 당뇨 대상 시험 번호 목록 (앞 5개만)
SELECT nct_id
FROM browse_conditions
WHERE mesh_term = 'Diabetes Mellitus, Type 2'
LIMIT 5;

In [ ]:
%%sql
-- 제2형 당뇨 대상 시험이 몇 건이야? → 1,353
SELECT COUNT(DISTINCT nct_id) AS 시험수
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Diabetes Mellitus, Type 2'
);

### 서브쿼리와 JOIN — 언제 무엇을

같은 답이 나오지만 쓰임이 다릅니다.

| | 서브쿼리 (IN) | JOIN |
|---|---|---|
| 무엇을 하나 | 대상 목록을 만들어 걸러냄 | 두 표를 옆으로 이어 붙임 |
| 행 수가 늘어나나 | 늘지 않음 (원래 표 그대로) | 1 : N 이면 늘어남 |
| 언제 쓰나 | ‘이 조건에 맞는 대상만’ 고를 때 | 다른 표의 **열을 함께 보여줄** 때 |
| 이번 사례에서는 | 대상 시험 고르기 | 상태 · 국가를 함께 보기 |

같은 질문 “한국 기관이 참여한 시험은 몇 건인가?” 를 두 방법으로 세어 **행 수**를 비교해 봅니다 (8주차 ‘자주 하는 실수 3’ 의 예제).

In [ ]:
%%sql
-- 서브쿼리 → 행수 1,182 (행 수 = 시험 수)
SELECT COUNT(*) AS 행수
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM facilities
    WHERE country = 'South Korea'
);

In [ ]:
%%sql
-- JOIN → 행수 4,098 / 시험수 1,182 (한 시험이 한국 기관 여러 곳에서 진행)
SELECT COUNT(*) AS 행수,
       COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN facilities AS f ON s.nct_id = f.nct_id
WHERE f.country = 'South Korea';

대상을 고르기만 할 때는 서브쿼리가 안전합니다. **행 수가 변하지 않기** 때문입니다.

## 2. 종합 예제 — 질환별 임상시험 현황

제2형 당뇨병 임상시험 현황을 구해 봅니다. 질문 하나를 네 단계로 나누어, 한 단계씩 더해 갑니다.

### 1단계 — 대상 고르기

**질문** 제2형 당뇨병을 대상으로 한 임상시험은 몇 건인가?

**확인** `COUNT(DISTINCT nct_id)` 를 쓴 이유는? 한 시험에 질환이 여러 개 붙어 있을 수 있기 때문입니다.

In [ ]:
%%sql
-- 1단계 → 1,353
SELECT COUNT(DISTINCT nct_id) AS 시험수
FROM browse_conditions
WHERE mesh_term = 'Diabetes Mellitus, Type 2';

### 2단계 — 진행 상태별로 나누기 (studies와 연결해 상태를 가져옵니다)

**질문** 그 시험들은 지금 어떤 상태인가?

**확인** 합계가 1,353인지 확인합니다. 맞으면 JOIN에서 빠진 시험이 없다는 뜻입니다.

In [ ]:
%%sql
-- 2단계 → 9종 (COMPLETED 817, UNKNOWN 243, ...)
SELECT s.overall_status AS 상태,
       COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b
  ON s.nct_id = b.nct_id
WHERE b.mesh_term = 'Diabetes Mellitus, Type 2'
GROUP BY s.overall_status
ORDER BY 시험수 DESC;

### 3단계 — 비율 구하기 (분모와 분자를 각각 정합니다)

**질문** 완료된 시험 중 결과가 게시된 비율은?

**확인** `CASE WHEN 조건 THEN 값 END` 은 “조건에 맞을 때만 값을 남긴다”는 뜻입니다. 191 ÷ 817 = 23.4%

In [ ]:
%%sql
-- 3단계 → 817 / 191
SELECT COUNT(DISTINCT s.nct_id) AS 완료시험,
       COUNT(DISTINCT CASE WHEN s.results_first_posted_date
             IS NOT NULL THEN s.nct_id END) AS 결과게시
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
WHERE b.mesh_term = 'Diabetes Mellitus, Type 2'
  AND s.overall_status = 'COMPLETED';

### 4단계 — 우리나라 참여 확인 (표를 하나 더 연결합니다)

**질문** 그 중 한국에서 진행된 시험은 몇 건인가?

**확인** 표를 두 개 연결했지만 `COUNT(DISTINCT nct_id)` 를 썼으므로 시험 수가 유지됩니다.

In [ ]:
%%sql
-- 4단계 → 76
SELECT COUNT(DISTINCT s.nct_id) AS 한국참여
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
JOIN countries AS co ON s.nct_id = co.nct_id
WHERE b.mesh_term = 'Diabetes Mellitus, Type 2'
  AND co.name = 'South Korea'
  AND co.removed = false;

### 여기까지 정리

| 전체 | 대상 질환 | 상태별 | 비율 | 한국 |
|---:|---:|---:|---:|---:|
| 37,509 | 1,353 | 9종 | 23.4% | 76 |

1. 질문을 **대상 · 기간 · 조건 · 단위**로 나누어 적는다
2. 조건을 **한 번에 하나씩** 더하며 건수를 확인한다
3. 표를 연결할 때마다 **COUNT(DISTINCT 키)** 로 단위를 지킨다

**잠깐 생각해 봅시다** 1단계에서 1,353건이었는데 2단계 합계가 1,340건이면 무엇을 의심해야 할까요?
→ JOIN에서 studies에 없는 시험이 빠졌을 수 있습니다. LEFT JOIN으로 다시 확인합니다 (4절 검산 ③).

## 3. 병원 SQL 사례 — 실제로 이렇게 씁니다

| 누가 | 무엇을 | 어떤 SQL이 쓰이나 |
|---|---|---|
| QI · 적정성평가 담당 | 지표별 대상자 수와 충족률 | 조건으로 대상자 선별 → 분모 · 분자 각각 COUNT |
| 감염관리팀 | 월별 감염 발생 건수, 재원일수 | GROUP BY 월 · 병동, 날짜 차이 계산 |
| 간호부 | 간호 인력 대비 환자 수, 낙상 건수 | GROUP BY 병동 · 근무조, COUNT와 비율 |
| 연구자 | 연구 대상자 선별, 변수 추출 | 여러 표를 JOIN하여 코호트 추출 |
| 원무 · 경영 | 진료과별 실적, 수납 현황 | GROUP BY 진료과 · 기간, SUM과 AVG |

공통점: **대상을 고르고(WHERE) → 묶어서 세고(GROUP BY) → 검산한다**. 오늘 배운 그대로입니다.

코호트 = 같은 조건으로 묶어 분석 대상으로 삼은 집단 (13주차에서 자세히).

> **병원 데이터라면** — 30일 재입원율: 분모는 지난달 퇴원한 입원 건(사망 · 전원 제외), 분자는 그 퇴원일로부터 30일 이내 재입원(계획된 재입원 제외)입니다. 어려운 부분은 SQL 문법이 아니라 **제외 기준을 정하는 일**입니다 (13주차).

## 4. 검산 — 결과를 믿기 전에

결과를 보고서에 쓰기 전에 **반드시 확인하는 다섯 가지**입니다.

| 확인할 것 | 무엇을 보는가 | 어떻게 |
|---|---|---|
| ① 전체 건수 | 조건 없이 센 값과 비교했을 때 말이 되는가 | `COUNT(*) FROM studies` |
| ② 세는 단위 | 시험 수인가, 행 수인가 | `COUNT(*)` 인가 `COUNT(DISTINCT nct_id)` 인가 |
| ③ 분모 | JOIN 때문에 대상이 줄지 않았는가 | `LEFT JOIN` 으로 다시 세기 |
| ④ 빠진 값 | NULL이 많은 열을 조건에 썼는가 | `COUNT(열)` 로 값 있는 행 세기 |
| ⑤ 이상한 값 | 최소 · 최대가 상식적인가 | `MIN()`, `MAX()` 로 확인 |

2절 종합 예제(제2형 당뇨병)의 결과를 다섯 가지로 점검해 봅니다.

In [ ]:
%%sql
-- ① 전체 건수 → 37,509. 대상 1,353건은 이 안에서 말이 되는 숫자입니다
SELECT COUNT(*) AS 전체
FROM studies;

In [ ]:
%%sql
-- ② 세는 단위 → 행수 1,353 = 시험수 1,353
-- 이번에는 같지만 기관 표처럼 달라지는 경우(1절: 4,098 vs 1,182)가 있어 COUNT(DISTINCT nct_id) 를 습관처럼 씁니다
SELECT COUNT(*) AS 행수,
       COUNT(DISTINCT nct_id) AS 시험수
FROM browse_conditions
WHERE mesh_term = 'Diabetes Mellitus, Type 2';

In [ ]:
%%sql
-- ③ 분모 → 대상시험 1,353 = 연결된시험 1,353 (JOIN에서 빠진 시험 없음)
SELECT COUNT(DISTINCT b.nct_id) AS 대상시험,
       COUNT(DISTINCT s.nct_id) AS 연결된시험
FROM browse_conditions AS b
LEFT JOIN studies AS s ON b.nct_id = s.nct_id
WHERE b.mesh_term = 'Diabetes Mellitus, Type 2';

In [ ]:
%%sql
-- ④ 빠진 값 → 817 / 191. COUNT(열) 은 값이 있는 행만 셉니다 (3단계와 다른 방법으로 세어도 같음)
SELECT COUNT(*) AS 완료시험,
       COUNT(results_first_posted_date) AS 결과게시일있음
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Diabetes Mellitus, Type 2'
)
  AND overall_status = 'COMPLETED';

In [ ]:
%%sql
-- ⑤ 이상한 값 → 최소 0명, 최대 4,000,000명. 상식적인가?
SELECT MIN(enrollment) AS 최소인원,
       MAX(enrollment) AS 최대인원
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Diabetes Mellitus, Type 2'
);

⑤ 0명 = 시작 전에 철회(WITHDRAWN)된 시험, 400만 명 = 덴마크 전국 단위 관찰연구 1건입니다. 평균을 내기 전에 이런 값부터 봅니다.

숫자가 틀리는 것보다 **틀린 줄 모르는 것**이 더 위험합니다. 12주차에서 이 다섯 가지를 오류 사례와 함께 다시 봅니다.

## 5. 과제 1 — 제출 안내 (잘못된 SQL 고치기, 20점)

과제는 따로 받는 `과제1.ipynb` 에서 합니다. 먼저 아래 ‘직접 해보기’ 로 오늘 순서를 연습해 보세요.

| 항목 | 내용 |
|---|---|
| 하는 일 | 동료가 쓴 SQL 12개는 실수가 한 군데씩 있어 **그대로 실행하면 오류가 납니다**. 오류 메시지를 읽고 그 한 군데를 고쳐 ✅ 가 나오면 1점(자동 채점)이고, 무엇이 왜 틀렸는지 한 줄로 적습니다. 마지막 해석 2문항은 문장으로 씁니다 |
| 채점 | 자동 확인 12점 (✅ 하나에 1점) · 결과 해석 5점 · 실행 완료 · 형식 3점 (문항마다 ‘무엇이 왜 틀렸나’ 한 줄 포함) |
| 범위 | 7 · 8주차 범위 — ① 조회 기초 · ② 집계와 GROUP BY · ③ 당뇨병 시험 분석 (JOIN 으로 두 표를 연결해 세기) |
| 분석 질환 | 모든 학생이 같은 질환(제2형 당뇨병)을 분석합니다. 숫자가 다르면 조건을 다시 확인하세요. |
| 제출 | 학번 · 이름을 첫 셀에 기입하고 모든 셀을 실행한 상태로 저장 → 파일 이름을 `과제1_학번_이름.ipynb` 로 바꿔 **9주차 수업 후 1주일 이내 LMS** |

막히면: 오류는 붉은 글씨 중 `Error:` 가 들어간 줄만 읽으면 됩니다 · `AS` 로 붙인 열 이름은 바꾸지 마세요 · 0행이면 대소문자와 따옴표 확인 · 데이터 파일이 안 열리면 런타임 다시 시작 후 처음부터. 7 · 8주차 실습 노트북의 ‘자주 하는 실수’ 를 다시 보세요.

## 직접 해보기 — 5문항 (위암으로 같은 단계 반복)

질환만 **위암**(MeSH 용어 `'Stomach Neoplasms'`)으로 바꾸어 오늘의 순서를 그대로 따라 합니다. `___` 를 채우고 실행한 뒤, **바로 다음 셀**의 `check(...)` 를 실행하세요. 두 번 틀리면 힌트가 나옵니다.
결과 열의 이름(`AS ...`)은 적힌 그대로 둡니다.

| 다섯 단계 | 위암 질문 |
|---|---|
| ① 질문을 한 문장으로 | 위암을 대상으로 한 임상시험은 몇 건이고, 상태 · 결과 게시 · 한국 참여는 어떤가? |
| ② 세는 단위 | 시험 건수 (한 행 = 시험 하나) |
| ③ 표와 연결 키 | `browse_conditions`(질환) · `studies`(상태, 결과 게시일) · `countries`(국가), 모두 `nct_id` 로 연결 |
| ④ 조건을 하나씩 | Q5 (위암 → + 중재 연구 → + 완료) |
| ⑤ 검산 | Q2 상태별 시험수의 합계가 Q1 과 같은가? |

**Q1. 서브쿼리** — 위암을 대상으로 한 시험이 몇 건인지 서브쿼리로 세세요. 괄호 안은 대상 시험 번호 목록, 바깥은 그 목록에 드는 시험만 셉니다. (1절 서브쿼리 예제)

In [ ]:
%%sql
-- Q1
SELECT COUNT(DISTINCT nct_id) AS 시험수
FROM studies
WHERE nct_id ___ (
    SELECT ___
    FROM browse_conditions
    WHERE mesh_term = 'Stomach Neoplasms'
);

In [ ]:
check("Q1", _)

**Q2. 2단계 — 상태별로 나누기** — studies 와 browse_conditions 를 연결해 위암 시험의 진행 상태별 시험 수를 구하세요. 두 표를 잇는 조건(연결 키)을 채웁니다. 실행한 뒤 시험수의 합계가 Q1 과 같은지 확인해 보세요.

In [ ]:
%%sql
-- Q2
SELECT s.overall_status AS 상태,
       COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b
  ON ___
WHERE b.mesh_term = 'Stomach Neoplasms'
GROUP BY s.overall_status
ORDER BY 시험수 DESC;

In [ ]:
check("Q2", _)

**Q3. 3단계 — 분모와 분자** — 완료된 위암 시험 수(`완료시험`, 분모)와 그중 결과가 게시된 시험 수(`결과게시`, 분자)를 구하세요. `CASE WHEN` 의 조건을 채웁니다.

In [ ]:
%%sql
-- Q3
SELECT COUNT(DISTINCT s.nct_id) AS 완료시험,
       COUNT(DISTINCT CASE WHEN s.results_first_posted_date
             ___ THEN s.nct_id END) AS 결과게시
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
WHERE b.mesh_term = 'Stomach Neoplasms'
  AND s.overall_status = 'COMPLETED';

In [ ]:
check("Q3", _)

**Q4. 4단계 — 우리나라 참여** — 위암 시험 중 한국에서 진행된 시험은 몇 건인가요? countries 표를 하나 더 연결했습니다. 새로 붙인 표(`co`)의 나라 이름 조건을 채웁니다.

In [ ]:
%%sql
-- Q4
SELECT COUNT(DISTINCT s.nct_id) AS 한국참여
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
JOIN countries AS co ON s.nct_id = co.nct_id
WHERE b.mesh_term = 'Stomach Neoplasms'
  AND co.name = ___
  AND co.removed = false;

In [ ]:
check("Q4", _)

**Q5. 조건을 하나씩 더하기** — “위암 시험 중 **중재 연구**이면서 **완료된** 것은 몇 건인가?” 한 번에 다 쓰지 않고, 조건을 하나 더할 때마다 건수를 적어 둡니다.

| 단계 | 조건 | 건수 |
|---|---|---|
| ① | 위암 대상 | Q1 의 결과 |
| ② | + 중재 연구 | 바로 아래 예제 셀 |
| ③ | + 완료 | Q5 |

먼저 아래 예제 셀(② + 중재 연구)을 실행해 건수를 보세요.

In [ ]:
%%sql
-- ② 위암 대상 + 중재 연구 → 몇 건?
SELECT COUNT(*) AS 시험수
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Stomach Neoplasms'
)
  AND study_type = 'INTERVENTIONAL';

이제 Q5 셀의 `___` 자리에 **완료(COMPLETED) 조건 한 줄**을 더해 ③의 건수를 구하세요. 건수가 ② 보다 줄었는지 확인합니다.

In [ ]:
%%sql
-- Q5
SELECT COUNT(*) AS 시험수
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Stomach Neoplasms'
)
  AND study_type = 'INTERVENTIONAL'
  ___

In [ ]:
check("Q5", _)

### 더 해보기 (선택)

**Q6. 검산 ⑤ 이상한 값** — 위암 시험의 등록 인원 최소(`최소인원`)와 최대(`최대인원`)를 구하세요. 괄호 안(대상 시험 번호 목록)을 직접 적습니다. 결과가 상식적인지 생각해 보세요. 최소 인원은 Q2 의 어느 상태와 관련이 있을까요?

In [ ]:
%%sql
-- Q6
SELECT MIN(enrollment) AS 최소인원,
       MAX(enrollment) AS 최대인원
FROM studies
WHERE nct_id IN (
    ___
);

In [ ]:
check("Q6", _)

## 오늘 배운 것

- 질문을 **대상 · 기간 · 조건 · 세는 단위**로 나누어 적은 뒤 SQL을 쓴다
- 대상만 고를 때는 **서브쿼리**, 다른 표의 열까지 볼 때는 **JOIN**
- 조건을 **하나씩** 더하며 건수가 어떻게 변하는지 본다
- **검산 다섯 가지**를 확인한 뒤에 숫자를 내보낸다

다음 시간: 10강. 안전한 데이터 수정과 트랜잭션 — INSERT · UPDATE · DELETE

아래 셀을 실행하면 오늘 확인한 문항(Q1~Q5)을 한눈에 볼 수 있습니다. 선택 문항 Q6 은 ‘(선택)’ 으로 표시되고 개수에는 들어가지 않습니다.

In [ ]:
진행확인()